# HW 11: Wind and chlorophyll panels, 1997–2016

In [1]:
import os
import gc
import dask
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.colors import LogNorm, Normalize
import matplotlib.dates as mdates
import cartopy
import cartopy.crs as ccrs
import xarray as xr
import cdsapi

# Run one Dask task at a time to reduce out-of-memory crashes.
#dask.config.set(scheduler='synchronous')

# import the moviepy module
import moviepy.video.io.ImageSequenceClip

## Folders

In [2]:
# Run this notebook with its own folder as the working directory.
lab_directory = os.getcwd()
era5_folder = os.path.join(lab_directory, 'era5')
chla_folder = os.path.join(lab_directory, 'chla-occci-v6-1km-1997-2016')
era5_dir = os.path.join(lab_directory, 'era5-panels')
chla_dir = os.path.join(lab_directory, 'chla-panels-temporal-7d')
for folder in [era5_folder, chla_folder, era5_dir, chla_dir]:
    os.makedirs(folder, exist_ok=True)

cutout_extent = (-120.5, -119.1, 33.85, 34.55)
west, east, south, north = cutout_extent

## Download chlorophyll
NOAA OCCCI v6

In [3]:
noaa_url = 'https://comet.nefsc.noaa.gov/erddap/griddap/occci_v6_daily_1km'
ds = xr.open_dataset(noaa_url, engine='netcdf4')
chl = ds[['chlor_a']].sel(
    time=slice('1997-11-01', '2016-03-01'),
    latitude=slice(35, 31.5), longitude=slice(-121, -116.5)
)
for day in chl.time.values:
    filename = os.path.join(chla_folder, f'chla_{str(day)[:10]}.nc')
    if os.path.exists(filename):
        continue
    print('Downloading', str(day)[:10])
    daily = chl.sel(time=[day]).rename({'chlor_a': 'CHL'}).sortby('latitude')
    daily.to_netcdf(filename)
ds.close()

## Download wind
Hourly ERA5 u/v, January–April and November–December, 1997–2016

In [4]:
dataset = 'reanalysis-era5-single-levels'
years = range(1997, 2017)
client = cdsapi.Client()
for year in years:
    target = os.path.join(era5_folder, f'wind_{year}.nc')
    if os.path.exists(target):
        continue
    request = {
        'product_type': ['reanalysis'],
        'variable': ['10m_u_component_of_wind', '10m_v_component_of_wind'],
        'year': [str(year)],
        'month': ['01', '02', '03', '04', '11', '12'],
        'day': [f'{day:02d}' for day in range(1, 32)],
        'time': [f'{hour:02d}:00' for hour in range(24)],
        'data_format': 'netcdf',
        'download_format': 'unarchived',
        'area': [35, -121, 31.5, -116.5],
    }
    client.retrieve(dataset, request).download(target)

2026-11-05 19:47:41,833 INFO [2026-10-06T00:00:00Z] **Data Stores Service protection measures** affecting data downloads and account access: [please read our Announcement](https://forum.ecmwf.int/t/data-stores-service-protection-measures-affecting-data-downloads-and-account-access-cds-ads-ewds-ecds/15481) before enquiring with ECMWF Support.


## Interpolate chlorophyll and calculate daily wind
Fill chlorophyll gaps between observations at most seven days apart, then use the filled data for both maps and the regional mean.

In [5]:
files = sorted(
    os.path.join(chla_folder, name)
    for name in os.listdir(chla_folder) if name.endswith('.nc')
)

interpolated_file = os.path.join(lab_directory, 'chla_temporal_7d.nc')

def interpolate_tile(tile):
    return tile.interpolate_na(
        dim='time', method='linear', max_gap='7D'
    )

if not os.path.exists(interpolated_file):
    with xr.open_mfdataset(
        files,
        combine='by_coords',
        chunks={'latitude': 100, 'longitude': 100},
    ) as raw_ds:
        raw_chla = raw_ds['CHL'].sel(
            time=slice('1997-11-01', '2016-03-01')
        ).chunk({'time': -1})

        filled_chla = raw_chla.map_blocks(
            interpolate_tile, template=raw_chla
        )

        temporary_file = interpolated_file + '.partial'
        write_task = filled_chla.to_dataset(name='CHL').to_netcdf(
            temporary_file,
            encoding={'CHL': {'chunksizes': (1, 100, 100)}},
            compute=False,
        )
        write_task.compute(scheduler='synchronous')

    os.replace(temporary_file, interpolated_file)

# Read individual days
chla_ds = xr.open_dataset(interpolated_file, chunks={'time': 1})
chla = chla_ds['CHL']

# Calculate the regional mean from interpolation
chla_mean = chla.sel(
    latitude=slice(south, north),
    longitude=slice(west, east),
).mean(dim=('latitude', 'longitude')).compute()

chla_mean = chla_mean.resample(time='1D').asfreq()

In [6]:
wind_files = [os.path.join(era5_folder, f'wind_{year}.nc')
              for year in range(1997, 2017)]
wind_ds = xr.open_mfdataset(wind_files, combine='by_coords')
hourly = wind_ds[['u10', 'v10']].rename({'valid_time': 'time'})
hourly['speed'] = np.sqrt(hourly['u10']**2 + hourly['v10']**2)
wind = hourly.resample(time='1D').mean().load().sortby('latitude')
wind_ds.close()

# Direction the wind comes from, clockwise from north (degrees).
wind['direction'] = (270 - np.degrees(np.arctan2(wind['v10'], wind['u10']))) % 360
wind['direction'] = wind['direction'].where((wind['u10'] != 0) | (wind['v10'] != 0))
wind_mean = wind['speed'].sel(
    latitude=slice(south, north), longitude=slice(west, east)
).mean(dim=('latitude', 'longitude'))

## Make panels
One PNG per available day in the requested date range. Wind arrows point in the direction the air moves.

In [7]:
def create_panel(start_date, end_date, mode):
    if mode == 'wind':
        data = wind['speed'].dropna('time', how='all')
        series = wind_mean
        output_dir = era5_dir
        label = 'Wind speed (m/s)'
        norm = Normalize(vmin=0, vmax=15)
    elif mode == 'chlorophyll':
        data = chla
        series = chla_mean
        output_dir = chla_dir
        label = 'Chl-a (mg m^-3)'
        norm = LogNorm(vmin=0.05, vmax=10)
    else:
        raise ValueError("mode must be 'wind' or 'chlorophyll'")

    data = data.sel(time=slice(start_date, end_date))
    series = series.sel(time=slice(start_date, end_date))
    image_list = []

    for day_time in data.time.values:
        date = day_time.astype('datetime64[s]').item()
        output_file = os.path.join(output_dir, mode + '_' + date.strftime('%Y%m%d') + '.png')
        # Keep existing images in the movie without loading or plotting them again.
        if os.path.exists(output_file):
            image_list.append(output_file)
            continue

        # Load each day once for both maps to reduce peak RAM usage.
        daily_data = data.sel(time=day_time).compute()
        lon = daily_data.longitude.values
        lat = daily_data.latitude.values

        data_crs = ccrs.PlateCarree()
        projection = ccrs.Mercator(central_longitude=-119)
        fig = plt.figure(figsize=(12, 9), layout='constrained')
        gs = fig.add_gridspec(3, 2, height_ratios=[3, 1, 0.3])
        ax = fig.add_subplot(gs[0, 0], projection=projection)
        ax_zoom = fig.add_subplot(gs[0, 1], projection=projection)
        ax_ts = fig.add_subplot(gs[1, :])
        ax.set_extent([-121, -116.5, 31.5, 35], crs=data_crs)
        ax_zoom.set_extent(cutout_extent, crs=data_crs)

        for map_ax in (ax, ax_zoom):
            C = map_ax.pcolormesh(
                lon, lat, daily_data.values, transform=data_crs,
                norm=norm, cmap='viridis'
            )
            if mode == 'wind':
                day_wind = wind.sel(time=day_time)
                map_ax.quiver(
                    lon, lat, day_wind['u10'].values, day_wind['v10'].values,
                    transform=data_crs, scale=100, width=0.003
                )
            map_ax.add_feature(
                cartopy.feature.LAND, zorder=99,
                edgecolor='black', facecolor='silver'
            )

        west, east, south, north = cutout_extent
        ax.add_patch(Rectangle(
            (west, south), east - west, north - south,
            fill=False, edgecolor='crimson', linewidth=2,
            transform=data_crs, zorder=100
        ))
        ax.set_title('Southern California Bight')
        ax_zoom.set_title('Santa Barbara Channel')
        cbar = fig.colorbar(C, ax=[ax, ax_zoom], shrink=0.65, aspect=13)
        cbar.set_label(label)
        fig.suptitle(mode.capitalize() + ' — ' + date.strftime('%Y/%m/%d'))

        visible = series.sel(time=slice(start_date, day_time))
        ax_ts.plot(visible.time.values, visible.values, color='teal', linewidth=1)
        ax_ts.axvline(day_time, color='crimson', linewidth=2, label='Current map date')
        ax_ts.set_title('Santa Barbara Channel — daily mean ' + mode)
        ax_ts.set_ylabel(label)
        ax_ts.set_xlim(np.datetime64(start_date) - np.timedelta64(1, 'D'),
                       np.datetime64(end_date) + np.timedelta64(1, 'D'))
        ax_ts.set_ylim(bottom=0)
        ax_ts.grid(alpha=0.25)
        ax_ts.legend(loc='upper left')
        locator = mdates.AutoDateLocator(minticks=4, maxticks=8)
        ax_ts.xaxis.set_major_locator(locator)
        ax_ts.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))

        month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                       'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
        ax2 = fig.add_subplot(gs[2, :])
        ax2.barh(0, date.month, height=0.7, color='teal')
        ax2.set_xlim(0, 12)
        ax2.set_ylim(-0.5, 0.5)
        ax2.set_xticks(np.arange(12) + 0.5, labels=month_names)
        ax2.set_yticks([])
        ax2.set_xlabel(f'Month ({date.year})')

        fig.savefig(output_file)
        plt.close(fig)
        image_list.append(output_file)
        # Garbage collection to prevent out-of-memory crash
        gc.collect()

    return image_list

## Generate wind and chlorophyll images in parallel

In [8]:
def render_mode(mode):
    plt.switch_backend('Agg')  # Save images without opening plot windows
    # Load one day at a time within each process to limit RAM use
    with dask.config.set(scheduler='synchronous'):
        return create_panel('1997-11-01', '2016-11-02', mode=mode)

# Use separate processes for speed
wind_images, chla_images = dask.compute(
    dask.delayed(render_mode)('wind'),
    dask.delayed(render_mode)('chlorophyll'),
    scheduler='processes',
    num_workers=2,
    chunksize=1,
)

# Make movies

In [9]:
# Set how many images are shown per second.
fps=10
image_array = [wind_images, chla_images]
for image_list in image_array:
    # use the ImageSequenceClip module to set up the clip
    clip = moviepy.video.io.ImageSequenceClip.ImageSequenceClip(image_list, fps=fps)

    # write the video to a file
    if image_list == wind_images:
        clip.write_videofile(os.path.join(lab_directory,'scb_wind.mp4'))
    else:
        clip.write_videofile(os.path.join(lab_directory,'scb_chla.mp4'))

Moviepy - Building video /home/eli2883/ms274/hw11/scb_wind.mp4.
Moviepy - Writing video /home/eli2883/ms274/hw11/scb_wind.mp4



Moviepy - Done !
Moviepy - video ready /home/eli2883/ms274/hw11/scb_wind.mp4
Moviepy - Building video /home/eli2883/ms274/hw11/scb_chla.mp4.
Moviepy - Writing video /home/eli2883/ms274/hw11/scb_chla.mp4



Moviepy - Done !
Moviepy - video ready /home/eli2883/ms274/hw11/scb_chla.mp4
